In [3]:
!pip uninstall -y torch torchvision torchaudio -q

!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 7.4 MB/s eta 0:00:00


In [1]:
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

print("Libraries imported successfully")

Libraries imported successfully


In [2]:
data = {

    "text": [
        "The transformer model achieved excellent accuracy.",
        "Large Language Models are revolutionizing AI.",
        "The football team won the championship.",
        "The cricket match was exciting.",
        "Neural networks are widely used in deep learning.",
        "The player scored a brilliant goal.",
        "Machine learning improves decision making.",
        "The tennis tournament starts tomorrow."
    ],

    "label": [
        1,
        1,
        0,
        0,
        1,
        0,
        1,
        0
    ]
}


dataset = Dataset.from_dict(data)


print(dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 8
})


In [3]:
print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

print("Tokenizer loaded")

Loading tokenizer...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded


In [4]:
def tokenize(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


dataset = dataset.map(tokenize)


dataset.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "label"
    ]
)


print("Tokenization completed")

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Tokenization completed


In [5]:
print("Loading BERT model...")


model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)


print("Model loaded")

Loading BERT model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded


In [6]:
training_args = TrainingArguments(

    output_dir="./fine_tuned_model",

    per_device_train_batch_size=2,

    num_train_epochs=3,

    logging_steps=1,

    save_strategy="no",

    report_to="none"

)


print("Training configuration completed")

Training configuration completed


In [7]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=dataset

)


print("Training Started...\n")


trainer.train()


print("\nTraining Completed")

Training Started...



Step,Training Loss
1,0.633670
2,0.664107
3,0.793781
4,0.662898
5,0.488700
6,0.420522
7,0.367584
8,0.597531
9,0.475994
10,0.322096



Training Completed


In [8]:
trainer.save_model("./fine_tuned_model")

tokenizer.save_pretrained(
    "./fine_tuned_model"
)


print("Model saved successfully")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully


In [9]:
classifier = pipeline(

    "text-classification",

    model="./fine_tuned_model",

    tokenizer="./fine_tuned_model"

)


print("Classifier loaded")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Classifier loaded


In [10]:
text = "Generative AI models improve intelligent automation."


result = classifier(text)


labels = {

    "LABEL_0": "Sports",

    "LABEL_1": "Technology"

}


print("\nPrediction")
print("-------------------------")

print("Input :", text)

print(
    "Predicted Class :",
    labels[result[0]["label"]]
)

print(
    "Confidence Score :",
    round(result[0]["score"],3)
)


Prediction
-------------------------
Input : Generative AI models improve intelligent automation.
Predicted Class : Technology
Confidence Score : 0.698
